# Integration controls

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lldddv2/relatipy/blob/main/docs/tutorials/03-integration-controls.ipynb)

This tutorial shows:

- how to choose the integration method, tolerances and step limits of
  `Orbit.solve` and `Orbit.integrate`;
- how to read the outcome of an integration (`IntegrationInfo`, `status`,
  `Termination`, exceptions and warnings);
- how `reset` and `copy` manage the mutable state of an `Orbit`.

All examples use a Kerr black hole of $10^6\,M_\odot$ with spin $0.9$ and a
stable bound orbit with semi-latus rectum $p = 6\,r_s$, where $r_s = 2GM/c^2$ is
the Schwarzschild radius, eccentricity $e = 0.3$ and inclination parameter
$x = 0.8$.

In [ ]:
%pip install -Uq relatipy

In [1]:
import warnings

import numpy as np
from astropy import units as u
from astropy.constants import c

import relatipy as rp

## 1. Black hole, orbit and time scale

The native solver works in mass-normalized units: lengths are divided by
$GM/c^2 = r_s/2$ and times by $GM/c^3 = r_s/(2c)$. We express all integration
spans as multiples of the Schwarzschild light-crossing time $T_s = r_s/c$ so
the example stays short.

In [2]:
bh = rp.Kerr(mass=1e6 * u.Msun, spin=0.9)
Ts = (bh.r_s / c).to(u.s)        # 2 G M / c^3
orbit = bh.orbit(p=(6 * bh.r_s).to(u.km), e=0.3, x=0.8)
tau_end = 1000 * Ts

print("r_s =", bh.r_s.to(u.km))
print("Ts  =", Ts)
print("tau_end =", tau_end)

r_s = 2953250.07610025 km
Ts  = 9.850981895282535 s
tau_end = 9850.981895282535 s


## 2. Default settings and `IntegrationInfo`

`Orbit.solve` integrates from the saved initial conditions and returns an
immutable `Solution`. Its `integration` attribute is an `IntegrationInfo`
record with the effective method, tolerances, step limits and counters.

With the defaults (`method="radau"`, `rtol=None`, `atol=None`), RelatiPy
selects `rtol = 1e-10` and an `atol` array of shape `(8,)` equal to `rtol`
times a characteristic scale of each component of the native state
$(2ct/r_s, 2R/r_s, \Theta, \Phi, u^t, u^R, u^\Theta, u^\Phi)$ at the starting
point.

In [3]:
sol = orbit.solve(tau_span=(0 * u.s, tau_end))
info = sol.integration

print("method    :", info.method)
print("rtol      :", info.rtol)
print("atol      :", info.atol)
print("atol shape:", info.atol.shape)
print("first_step, max_step:", info.first_step, info.max_step)
print("n_steps, nfev:", info.n_steps, info.nfev)

method    : radau
rtol      : 1e-10
atol      : [2.80450902e-09 9.23076923e-10 1.00000000e-10 1.00000000e-10
 1.21994127e-10 3.29140294e-11 3.56568652e-12 5.78469462e-12]
atol shape: (8,)
first_step, max_step: None None
n_steps, nfev: 390 21157


## 3. Choosing a method

Four case-sensitive method names are accepted:

- `"radau"`: the default, a three-stage, fifth-order implicit Radau IIA method;
- `"dop853"`: explicit Dormand--Prince 8(5,3);
- `"dp45"`: explicit Dormand--Prince 5(4);
- `"projection_radau"`: uses the Radau stepper and projects each accepted step
  toward the initial values of the timelike norm, energy, axial angular momentum
  and Carter constant.

All methods share the same
controls. The step and function-evaluation counts differ:

In [4]:
methods = ("radau", "dop853", "dp45", "projection_radau")
solutions = {}
for method in methods:
    s = orbit.solve(tau_span=(0 * u.s, tau_end), method=method)
    solutions[method] = s
    print(f"{method:17s} status={s.status}  n_steps={s.integration.n_steps:5d}  "
          f"nfev={s.integration.nfev:6d}")

radau             status=0  n_steps=  390  nfev= 21157
dop853            status=0  n_steps=  250  nfev=  3770
dp45              status=0  n_steps= 1195  nfev=  7172
projection_radau  status=0  n_steps=  390  nfev= 21157


An unknown method name (including a different capitalization) is rejected
before integration with a `ValueError`:

In [5]:
try:
    orbit.solve(tau_span=(0 * u.s, tau_end), method="RADAU")
except ValueError as exc:
    print("ValueError:", exc)

ValueError: method must be one of radau, dop853, dp45, projection_radau


## 4. Explicit tolerances

`rtol` and `atol` are resolved independently. Supplying only `rtol` still
scales the automatic `atol`; supplying a scalar `atol` uses it for all eight
components. The effective values are always recorded in `IntegrationInfo`.

In [6]:
tight = orbit.solve(tau_span=(0 * u.s, tau_end), rtol=1e-12)
print("rtol=1e-12 -> atol =", tight.integration.atol)

scalar_atol = orbit.solve(tau_span=(0 * u.s, tau_end), atol=1e-12)
print("atol=1e-12 -> stored atol =", scalar_atol.integration.atol)

rtol=1e-12 -> atol = [2.80450902e-11 9.23076923e-12 1.00000000e-12 1.00000000e-12
 1.21994127e-12 3.29140294e-13 3.56568652e-14 5.78469462e-14]
atol=1e-12 -> stored atol = 1e-12


### 4.1 `IntegrationWarning`

Explicit tolerances are used as given, but an `IntegrationWarning` is issued
before integrating when they look unfit for the starting state:

- `rtol > 1e-6`;
- `0 < rtol < 100 * eps`;
- an `atol` entry larger than `1e-6` times the characteristic scale of
  $2ct/r_s$, $2R/r_s$, $\Theta$, $\Phi$ or $u^t$.

The
warning does not change the tolerances, the status or the exceptions. Omitted
tolerances never trigger it.

In [7]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", rp.IntegrationWarning)
    loose = orbit.solve(tau_span=(0 * u.s, tau_end), rtol=1e-3)

for w in caught:
    print(f"{w.category.__name__}: {w.message}")
print("stored rtol:", loose.integration.rtol)
print("status:", loose.status)
print("message:", loose.message)

stored rtol: 0.001
status: -1
message: Kerr integration failed before the target time (right-hand side evaluation failed; native status 4)


For this orbit the loose run did not reach the requested end: the warning was
a useful hint. Automatic tolerances are a heuristic based on the starting
state; they do not guarantee the global trajectory error either.

`solve` also warns when the spacing of requested `tau_eval` samples exceeds
one tenth of the Kepler period estimated from the initial osculating elements
(for a bound osculating conic). This is a Newtonian sampling estimate, not an
error bound.

In [8]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", rp.IntegrationWarning)
    sparse = orbit.solve(tau_eval=np.linspace(0, 1000, 3) * Ts)

for w in caught:
    print(f"{w.category.__name__}: {w.message}")

## 5. Step controls: `first_step` and `max_step`

Both are optional positive proper-time quantities (`None` lets the native
integrator choose). `max_step` caps every step, which raises the number of
steps; `first_step` only sets the size of the first trial step. Non-positive
or non-finite values raise `ValueError`.

In [9]:
for max_step in (None, 2.5 * Ts, 0.5 * Ts):
    s = orbit.solve(tau_span=(0 * u.s, tau_end), max_step=max_step)
    print(f"max_step={str(s.integration.max_step):>22s}  n_steps={s.integration.n_steps}")

s = orbit.solve(tau_span=(0 * u.s, tau_end), first_step=5e-4 * Ts)
print("first_step =", s.integration.first_step, " n_steps =", s.integration.n_steps)

try:
    orbit.solve(tau_span=(0 * u.s, tau_end), first_step=-1 * u.s)
except ValueError as exc:
    print("ValueError:", exc)

max_step=                  None  n_steps=390
max_step=   24.62745473820634 s  n_steps=476
max_step=   4.925490947641268 s  n_steps=2002
first_step = 0.004925490947641268 s  n_steps = 393
ValueError: first_step must be finite and positive


## 6. Comparing methods

Without `tau_eval`, a solution stores the initial state and every accepted
native step, so the differences of `solution.tau` are the accepted proper-time
steps $\Delta\tau$. The first table summarizes them for each method. The second
repeats each integration for several `rtol` values and prints the distance
between its final position and that of a `dop853` run with `rtol=1e-13`. This
is a self-consistency check between runs, not a measurement of the true global
error: tolerances control the local error only.

In [10]:
print(f"{'method':8s} {'n_steps':>8s} {'min dtau/Ts':>12s} {'median':>10s} {'max':>10s}")
for method in ("radau", "dop853", "dp45"):
    dtau = np.diff((solutions[method].tau / Ts).to_value(u.one))
    print(f"{method:8s} {dtau.size:8d} {dtau.min():12.3g} {np.median(dtau):10.3g} {dtau.max():10.3g}")

method    n_steps  min dtau/Ts     median        max
radau         390       0.0591        2.2        5.6
dop853        250       0.0245        3.4       8.86
dp45         1195      0.00402      0.683       2.13


In [11]:
ref = orbit.solve(tau_span=(0 * u.s, tau_end), method="dop853", rtol=1e-13)
rtols = (1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11)
methods = ("radau", "dop853", "dp45")

print("final-position difference from dop853, rtol=1e-13 [km]")
print(f"{'rtol':>8s}" + "".join(f"{m:>11s}" for m in methods))
for rtol in rtols:
    row = []
    for method in methods:
        run = orbit.solve(tau_span=(0 * u.s, tau_end), method=method, rtol=rtol)
        assert run.status == 0, (method, rtol, run.message)
        row.append(np.linalg.norm((run.xyz[-1] - ref.xyz[-1]).to_value(u.km)))
    print(f"{rtol:8.0e}" + "".join(f"{d:11.3e}" for d in row))

final-position difference from dop853, rtol=1e-13 [km]
    rtol      radau     dop853       dp45
   1e-06  1.901e+04  8.399e+03  3.964e+04
   1e-07  3.804e+03  3.633e+02  3.416e+03
   1e-08  6.565e+02  3.722e+01  3.642e+02
   1e-09  1.090e+02  3.657e+00  3.741e+01
   1e-10  1.634e+01  3.740e-01  3.582e+00
   1e-11  2.364e+00  1.808e-02  3.490e-01


## 7. Reading the outcome

Every `Solution` carries `status`, `success`, `message` and `termination`:

* `status == 0`: the requested end was reached (the last stored proper time is
  then exactly the requested end);
* `status == 1`: an internal terminal event stopped the solver, such as a
  confirmed crossing of the outer horizon. `termination` is then a
  `Termination` record with `reason` (`"outer_horizon"` for the horizon),
  `tau` and `state`, the last stored valid point outside the horizon;
* `status == -1`: numerical failure. The solution keeps the samples reached so
  far and `success` is `False`.

`success` is `True` for status `0` and `1`.

In [12]:
print("status     :", sol.status)
print("success    :", sol.success)
print("message    :", sol.message)
print("termination:", sol.termination)
print("last tau == tau_end:", sol.tau[-1] == tau_end)

status     : 0
success    : True
message    : Kerr integration reached the target time
termination: None
last tau == tau_end: True


### 7.1 A plunging orbit

A particle released from rest at $x = 5\,r_s$ falls toward the hole. In
Boyer--Lindquist coordinates the horizon is a coordinate singularity, and the
solver can fail before any accepted step crosses $R \le r_+$; that outcome is
reported as a numerical failure (`status == -1`), not as a terminal event.
The partial solution ends just outside the outer horizon:

In [13]:
plunge = bh.orbit(x=(5 * bh.r_s).to(u.km))
fall = plunge.solve(tau_span=(0 * u.s, 100 * Ts))

print("status :", fall.status)
print("success:", fall.success)
print("message:", fall.message)
print("samples:", len(fall))
print("last tau:", fall.tau[-1])
print("last R / r_s        :", (fall.R[-1] / bh.r_s).to(u.one))
print("outer horizon / r_s :", (bh.horizons.event / bh.r_s).to(u.one))
print("termination:", fall.termination)

status : -1
success: False
message: Kerr integration failed before the target time (right-hand side evaluation failed; native status 4)
samples: 325
last tau: 165.2513845386967 s
last R / r_s        : 0.7179449475015433
outer horizon / r_s : 0.7179449471770336
termination: None


Code that handles both kinds of early stop can branch on `status` and use
`termination` only when it is set:

In [14]:
def describe(solution):
    if solution.status == 1:
        t = solution.termination
        return f"terminal event {t.reason!r} at tau = {t.tau:.6g}"
    if solution.status == -1:
        return f"numerical failure after {len(solution)} samples"
    return "requested end reached"

print(describe(sol))
print(describe(fall))

requested end reached
numerical failure after 325 samples


### 7.2 Partial results with `tau_eval`

With `tau_eval`, a partial solution keeps only the requested samples reached
before the failure or event. If none of them was reached, `solve` raises
`IntegrationError` (numerical failure) or `IntegrationTerminated` (terminal
event) instead of returning unrequested samples.

In [15]:
partial = plunge.solve(tau_eval=np.linspace(0, 100, 11) * Ts)
print("status:", partial.status)
print("kept samples:", partial.tau)

try:
    plunge.solve(tau_eval=[150, 200] * Ts)
except (rp.IntegrationError, rp.IntegrationTerminated) as exc:
    print(f"{type(exc).__name__}: {exc}")

status: -1
kept samples: [ 0.         98.50981895] s
IntegrationError: Kerr integration failed before the target time (right-hand side evaluation failed; native status 4); no tau_eval sample was reached


## 8. `integrate`: a mutable current point

`Orbit.integrate(tau)` advances the orbit's current point to an absolute
proper time and returns `None`. Instead of a status code it raises:

* `IntegrationTerminated` for a terminal event, with `reason`, `tau`, `state`
  and the full `termination` record;
* `IntegrationError` for a numerical failure.

In both cases the orbit keeps its last valid point.

In [16]:
try:
    plunge.integrate(100 * Ts)
except rp.IntegrationTerminated as exc:
    print("terminated:", exc.reason, exc.tau)
except rp.IntegrationError as exc:
    print("IntegrationError:", exc)

print("current tau  :", plunge.tau)
print("current R/r_s:", (plunge.R / bh.r_s).to(u.one))

IntegrationError: Kerr integration failed before the target time (right-hand side evaluation failed; native status 4)
current tau  : 165.2513845386967 s
current R/r_s: 0.7179449475015433


## 9. `solve` versus `integrate`, `reset` and `copy`

* `solve` always starts from the saved initial conditions and leaves the orbit
  unchanged.
* `integrate` moves the current point. A target earlier than the current point
  restarts from the saved initial conditions and integrates forward; a target
  before the initial proper time is a `ValueError`.
* `reset()` restores the saved initial point without integrating.
* `copy()` returns an independently evolving orbit with the same initial and
  current points.

In [17]:
orbit.reset()
R0 = orbit.R                     # starting radius, before integrating
_ = orbit.solve(tau_span=(0 * u.s, tau_end))
print("after solve      : tau =", orbit.tau)

orbit.integrate(250 * Ts)
print("after integrate  : tau =", orbit.tau)

twin = orbit.copy()
twin.integrate(400 * Ts)
print("copy advanced    : orbit.tau =", orbit.tau)
print("                   twin.tau  =", twin.tau)

orbit.integrate(50 * Ts)        # earlier target: restarts from initial
print("earlier target   : tau =", orbit.tau)

orbit.reset()
print("after reset      : tau =", orbit.tau)
print("R unchanged      :", bool(orbit.R == R0))

try:
    orbit.integrate(-1 * u.s)
except ValueError as exc:
    print("ValueError:", exc)

after solve      : tau = 0.0 s
after integrate  : tau = 2462.745473820634 s
copy advanced    : orbit.tau = 2462.745473820634 s
                   twin.tau  = 3940.392758113014 s
earlier target   : tau = 492.5490947641268 s
after reset      : tau = 0.0 s
R unchanged      : True
ValueError: tau precedes the saved initial proper time


## Next steps

* [Constants of motion](07-constants-of-motion.ipynb): $E$, $L_z$ and $Q$ along a solution, and the drift of each method.
* [Solutions and states](04-solutions-and-states.ipynb): read coordinates,
  velocities and osculating elements from a `Solution`.
* [Plotting](05-plotting.ipynb): visualize trajectories.
* {doc}`../user-guide/usage` and {doc}`../user-guide/scope-and-limits`:
  tolerance rules, termination semantics and current limits.
* {doc}`../reference/geodesic`: API reference for `Orbit`, `Solution`,
  `IntegrationInfo` and the exceptions.